# Reporte de hallazgos — Réplica y actualización de Aradillas (2018)

**Insumo para white paper.** Documento de trabajo interno; los números provienen de
`aradillas_2014.ipynb`, `aradillas_2022.ipynb` y `comparacion_2014_2022.ipynb`.

*Última actualización: 5 de agosto de 2026.*

---

## Resumen

Replicamos en Python el estudio de Aradillas López (2018) para COFECE y lo actualizamos con
la ENIGH 2022. El ejercicio produjo tres tipos de resultado:

1. **Una réplica que valida el estudio en lo esencial** — el Gini observado coincide al
   decimal (0.481), las elasticidades regionales caen todas dentro de ±0.15, y los
   parámetros de poder de mercado quedan en el rango publicado.

2. **Discrepancias verificables entre lo que el paper declara y lo que su código hace**,
   ocho en total. Una de ellas —la ausencia de controles de costo por sector— compromete
   la identificación del parámetro central del estudio.

3. **Una actualización a 2022** que muestra que el costo del poder de mercado para los
   hogares creció, con evidencia sólida en los parámetros de markup y evidencia preliminar
   en las magnitudes de bienestar.

Sobre el acceso al código original: `programa_ENIGH_2014.g` se obtuvo por solicitud de
acceso a la información pública. **Nada de la sección 2 de este reporte habría sido
detectable leyendo únicamente el documento publicado.**

---

## Cómo leer este reporte

| sección | contenido | uso sugerido en el white paper |
|---|---|---|
| 1 | Qué replica y qué no | validación / metodología |
| **2** | **Discrepancias código ↔ paper** | **contribución principal** |
| 3 | El hallazgo sobre escalabilidad del estimador | contribución metodológica |
| 4 | Resultados 2022 y comparación | resultados |
| 5 | Limitaciones | honestidad metodológica |
| 6 | Agenda abierta | trabajo futuro |

---

# 1. Qué replica el ejercicio y qué no

| resultado | réplica | paper | veredicto |
|---|---|---|---|
| **Gini observado** | **0.481** | **0.481** | exacto |
| Gini contrafactual | 0.451 | 0.446 | ±0.005 |
| Cuadro 5 — elasticidades por región | 8/8 dentro de ±0.15 | | replica |
| Cuadro 8 — β_η | rango publicado (0.02–1.48) | | replica en estructura |
| Cuadro 4 — elasticidades nacionales | MAE 0.239, 10/13 dentro de ±0.30 | | parcial |
| Muestra final | 8,940 hogares | 15,586 | **no reproducible** |

**La brecha muestral no se explica.** El paper declara 15,586 hogares (≈80 % de la ENIGH) y
ninguna combinación de los filtros documentados la reproduce. Nuestro universo es de 12,372
antes del recorte iterativo y 8,940 después. Ver §2, divergencias B y C.

### Nueve errores de implementación corregidos

Durante la réplica se identificaron y corrigieron nueve errores **de nuestro propio código**
(no del estudio original), todos verificados contra el programa Gauss. Se listan porque
condicionan la lectura de versiones anteriores de este trabajo, incluidas las que Victor
produjo antes del refactor:

| # | error | efecto |
|---|---|---|
| N4 | columnas del archivo de precios leídas como contiguas | transporte aéreo a \$138 en vez de \$2,279 |
| **N8/N8b** | **pesos del índice Divisia sin sumar 1** | **elasticidades comprimidas a −1** |
| N10 | Gini sobre `ing_total` y muestra filtrada | Gini 0.448 en vez de 0.481 |
| N11 | denominador de la VE equivocado | VE/ingreso 10.0 % en vez de 15.8 % |
| N6, N12, N13, N14 | filtro de selección, `nanmean`, fallback a precios originales, solver | varios |

El detalle completo está en `CLAUDE.md`. **N8 es el relevante para la narrativa**: era la
causa de que las elasticidades salieran pegadas a −1 y los markups saturados, tanto en 2014
como en 2022.

---

# 2. Discrepancias entre el código y el documento publicado

**Ésta es la contribución principal del proyecto.** Ocho discrepancias verificadas, con su
ubicación exacta en el programa original.

## 2.1 Las cuatro que afectan la interpretación de los resultados

### D-H — Los controles de costo no son específicos por sector

> **El paper** (p. ~1377): *"variables de costos de insumos por empresa **específicos para
> aquellas ramas de actividad económica relacionadas con cada una de las doce categorías**
> de gasto"*. El **Cuadro 7** presenta el mapeo detallado a ramas SCIAN.

**El código** (l.6182–6209):

1. Carga `indicadores_costos_censos_economicos_2014.asc` como `[46,11]`: 46 filas = 46
   **ciudades**, 11 columnas de indicadores. **El archivo no tiene dimensión de sector.**
2. Las líneas 6197–6209 son **trece reasignaciones consecutivas e incondicionales** de
   `vars_costos`. Cada una sobrescribe la anterior: solo sobrevive la última. Las otras doce
   son código muerto.
3. El bloque está **dentro del bucle de categorías** (l.5931–6324) y se reejecuta idéntico
   en cada vuelta.

**Consecuencia:** tortillas, pan, carne de res, medicamentos y transporte aéreo comparten el
mismo vector de controles de costo, que son características de la **ciudad**, no del sector.

**Por qué compromete la conclusión central:** el modelo NEIO existe para separar la
variación de precios atribuible a costos de la atribuible a poder de mercado. Sin controles
de costo que varíen por sector, toda variación de precio específica de un sector se atribuye
a poder de mercado **por construcción**. El sesgo sobre `β_η` es al alza y sistemático.

### D-A — El estimador descrito no es el implementado

El paper describe una estimación en **dos etapas (OLS + GMM)**. El programa implementa
**únicamente el bucle OLS** de 16 iteraciones (l.2452–5753); no existe segunda etapa. Los
parámetros publicados son OLS.

### D-B — Un filtro de muestra no declarado

El código descarta los hogares que no son propietarios de su vivienda (l.1101), **el 27 % de
la muestra (5,215 hogares)**. El paper no lo menciona al describir el universo del estudio,
que declara construido solo con el criterio de distancia y el de relevancia de categorías.
El sesgo hacia propietarios no está discutido.

### D-D — El recorte iterativo no declarado

El programa recorta el 1 % de cada cola de la utilidad **en cada una de las 16 iteraciones**,
de forma acumulativa (l.5671–5722, dentro del bucle). Se pierde el **28 %** de la muestra:
12,372 → 8,940. El paper no lo menciona, y la muestra efectiva de estimación no es la que
reporta.

## 2.2 Las cuatro restantes

| # | discrepancia | ubicación |
|---|---|---|
| D-C | Los 15,586 hogares declarados no son reconstruibles con los criterios publicados | — |
| D-E | Un piso numérico (0.01) sobre gastos nulos contamina los pesos de los índices de precio: en transporte, el 71.5 % de los hogares recibe un reparto 50/50 inventado (real: 73/27); en carne de res, las vísceras pesan 22.8 % cuando su participación real es 3.4 % | l.1766 y análogas |
| D-F | La categoría "Pan" agrega producción industrial y panadería tradicional. **El CD incluye `precios_pan_de_caja_*.asc` y el programa nunca los usa** | l.1979-1990 |
| D-G | Tres correcciones verificadas **alejan** los resultados de las cifras publicadas | N4, N9, N13 |

### Sobre D-G

Corregir errores demostrables mueve el Gini contrafactual de 0.446 a 0.442, la VE/ingreso de
15.8 % a 17.4 % y el MAE de 0.218 a 0.251 — **en dirección contraria a lo publicado**. La
lectura natural es que las cifras del paper incorporan errores que se compensan entre sí.

Es un argumento sobre **robustez**, no sobre honestidad: no se puede hacer sin haber
replicado, y es la razón por la que el criterio de este proyecto es fidelidad al código y no
a los números publicados.

### Sobre D-F, y su relación con D-H

Pan es **sistemáticamente el sector con más poder de mercado medido**: β_η de 1.477 en el
paper (el 2º más alto), 1.011 en nuestra réplica y **1.496 en 2022** (el más alto, t=19.06).
Si esa categoría mezcla un mercado concentrado con uno atomizado, el resultado más citable
del estudio es un promedio de dos cosas distintas.

D-F es sobre la definición de la **categoría de gasto**; D-H es sobre el lado de los
**costos**. Son independientes y se acumulan.

---

# 3. Hallazgo metodológico: el estimador no escala

**El algoritmo del estudio original deja de funcionar cuando la encuesta crece.**

El recorte iterativo (D-D) es inocuo con la muestra de 2014 y destruye la identificación con
la de 2022. No es un error de programación: es una propiedad del estimador que **solo se
vuelve visible al actualizar el ejercicio con datos nuevos**.

| | 2014 (8,940 hog.) | 2022 (57,552 hog.) |
|---|---|---|
| varianza del regresor de utilidad | 1.170 | **0.450** |
| hogares con efectos ingreso planos | 6.9 % | **76.1 %** |
| convergencia del solver | ~97 % | **31.9 %** |
| rango de elasticidades | [0.69, 1.74] | **[0.04, 3.35]** |

**Mecanismo.** El recorte elimina las colas de la utilidad en cada iteración. Con 8,940
hogares esas colas se regeneran entre iteraciones y la varianza sobrevive; con 57,552 los
cuantiles son estables, el recorte muerde siempre en el mismo lugar y la varianza colapsa.
Sin varianza en el regresor, los coeficientes de efectos ingreso quedan sin identificar, la
función de costo se vuelve plana y el solver de utilidad no converge.

**Y el efecto es asimétrico:** desactivar el recorte *mejora* 2022 y *empeora* 2014
(MAE 0.217 → 0.286). El recorte no es un defecto en abstracto — está calibrado para un
tamaño de muestra concreto, sin que nada lo advierta.

**Implicación práctica:** cualquier actualización del estudio con ENIGH moderna (2022, 2024)
debe desactivar el recorte o recalibrarlo. Aplicarlo tal cual produce resultados
inservibles con apariencia de normalidad — que es exactamente lo que ocurrió en la primera
versión de nuestra actualización, con las 13 elasticidades pegadas a −1.

---

# 4. Resultados de la actualización 2022

## 4.1 Poder de mercado — el resultado sólido

| categoría | β 2014 | β 2022 | cambio | |
|---|---|---|---|---|
| **Materiales de construcción** | 0.009 | **0.515** | **+0.506** | gana significancia |
| **Pan** | 1.011 | **1.496** | **+0.485** | el más alto, t=19.06 |
| Tortillas | 0.119 | 0.349 | +0.230 | |
| Bebidas | 0.272 | 0.442 | +0.170 | |
| Carnes procesadas | 0.197 | 0.322 | +0.125 | gana significancia |
| Medicamentos | 0.310 | 0.406 | +0.097 | |
| Verduras | 0.502 | 0.136 | −0.366 | |
| Frutas | 0.966 | 0.685 | −0.281 | |
| Lácteos | 0.726 | 0.084 | −0.642 | pierde significancia |
| Pollo y huevo | 0.268 | 0.095 | −0.174 | pierde significancia |

*(Ambos años sin recorte, para que la comparación sea válida. 10 sectores significativos en
2014, 9 en 2022.)*

**Lectura:** el poder de mercado se **reconfigura** más que crecer parejo. Se concentra en
**materiales de construcción y pan** —vivienda y alimento básico— y retrocede en lácteos,
pollo y verduras.

## 4.2 Bienestar — dirección sólida, magnitudes preliminares

| decil | 2014 | 2022 |
|---|---|---|
| 1 (más pobre) | 14.0 % | **28.2 %** |
| 5 | 6.8 % | 13.5 % |
| 10 (más rico) | 2.6 % | 5.1 % |
| **Total** | **7.0 %** | **13.7 %** |
| Regresividad D1/D10 | 5.48 | 5.54 |
| Reducción del Gini | 3.1 % | **7.0 %** |

**La carga aproximadamente se duplica en todos los deciles, y la regresividad casi no
cambia.** No es que el "impuesto" se haya vuelto más desigual: creció parejo, y por eso
pesa mucho más abajo en términos absolutos.

**⚠️ Los niveles no son citables.** Ver §5.

---

# 5. Limitaciones

### 5.1 Las magnitudes de bienestar están sobreestimadas

La variación equivalente implica que eliminar el poder de mercado permitiría comprar la
misma canasta pagando el 23 % de lo que se paga (2022) o el 62 % (2014). No es creíble.

| | VE / gasto en categorías |
|---|---|
| implícito en el paper | ~0.28 |
| nuestra réplica 2014 | 0.379 |
| nuestra actualización 2022 | **0.767** |

**No es saturación de markups** — se verificó: 0.0 % de los markups toca el tope en 2014 y
1.7 % en 2022. El origen está en la evaluación de la función de costo, que usa
participaciones de gasto predichas **sin recortar a valores no negativos** y una definición
de utilidad distinta de la del resto del pipeline. Ambas inconsistencias están acotadas y
son corregibles; no se corrigieron aquí.

**Recomendación para el white paper:** usar los parámetros de markup (§4.1) como resultado
cuantitativo principal, y el bienestar (§4.2) como dirección y orden de magnitud, no como
cifra.

### 5.2 Categorías que no deben reportarse como cambio económico

* **Transporte foráneo** — su valor de 2014 en la comparación (0.169) proviene de la
  configuración sin recorte, que degrada precisamente esta categoría (con recorte: 0.800).
* **Carne de res** — se mueve 1.0 puntos entre años; es la categoría peor identificada en
  ambos, por D-E (las vísceras pesan 7× de más).
* **Bebidas** — mayor desviación respecto de 2014; candidata a estar afectada por el 34 % de
  hogares que conserva efectos ingreso débilmente identificados incluso sin recorte.

### 5.3 Comparabilidad entre años

* Los Gini de 2014 (0.481, sobre `ing_mon`) y 2022 (0.446) **no son comparables**: la ENIGH
  2022 "Nueva serie" dejó de publicar el ingreso monetario y hubo que reconstruirlo. La
  comparación de §4 usa `ing_cor`, única variable con definición idéntica en ambos años.
* La ENIGH 2022 tiene 90,102 hogares contra 19,124 en 2014. Las tasas de retención tras
  filtros son casi iguales (64.4 % y 65.8 %), pero los errores estándar de 2022 son
  mecánicamente menores.

### 5.4 Datos

Tres ciudades carecen de serie de INPC en el año base y usan el factor mediano nacional;
una de ellas es el Área Metropolitana de la Ciudad de México, el mercado más grande.

---

# 6. Agenda abierta

**Ordenada por relación entre esfuerzo y valor para el white paper.**

### Alta prioridad

1. **Separar Pan industrial de panadería tradicional.** "Pan de caja" existe como genérico
   propio en los precios de INEGI 2022, así que la prueba es directa: agregar una entrada a
   `CATEGORIAS` y reestimar. Si β_η del segmento industrial supera al agregado actual de
   1.496, D-F pasa de crítica metodológica a resultado cuantificado.

2. **Corregir la evaluación de la variación equivalente** (§5.1). Desbloquea todas las
   magnitudes de bienestar, que hoy son el resultado más visible y el menos defendible.

3. **Cuantificar el sesgo de D-H.** Construir controles de costo por sector desde los Censos
   Económicos y reestimar. Permitiría decir *cuánto* del `β_η` publicado es poder de mercado
   y cuánto es variación de costos no controlada.

### Media

4. Resolver la brecha muestral de 15,586 hogares (D-C).
5. Verificar si Bebidas y Transporte reflejan cambio económico o residuo de identificación.
6. Serie de INPC propia para el Área Metropolitana de la CDMX.

### Extensión

7. **ENIGH 2024.** La arquitectura ya lo contempla: escribir `datos_2024.cargar()` que
   devuelva un `DatosAnio`. El núcleo de cálculo no se toca.

---

# Anexo — Reproducibilidad

Todo lo citado se regenera con los notebooks del repositorio. Desde el directorio que
contiene `Replica_COFECE/`:

| notebook | produce | tiempo |
|---|---|---|
| `aradillas_2014.ipynb` | réplica 2014 (§1) | ~30 s |
| `aradillas_2022.ipynb` | actualización 2022 | ~6 min |
| `comparacion_2014_2022.ipynb` | comparación bajo tratamiento idéntico (§4) | ~7 min |

Los datos no están versionados: son microdatos públicos de INEGI. Sí lo está `CD/`, con el
programa Gauss original y los `.asc` preprocesados — **material irreemplazable**, obtenido
por solicitud de acceso a la información.

La celda siguiente verifica que el entorno esté completo y lista las referencias de línea
del Gauss usadas en §2, para quien quiera comprobarlas.

In [1]:
import os, sys
sys.path.insert(0, "Replica_COFECE/Codigo")

for m in ["aradillas_core", "datos_base", "datos_2014", "datos_2022"]:
    print(f"  {'ok ' if os.path.exists(f'Replica_COFECE/Codigo/{m}.py') else 'FALTA'} {m}.py")
gauss = "Replica_COFECE/CD/programa_ENIGH_2014.g"
print(f"  {'ok ' if os.path.exists(gauss) else 'FALTA'} {gauss}")

print("\nReferencias de línea en programa_ENIGH_2014.g (§2):")
for d, l in [("D-A  estimación en una sola etapa OLS", "2452-5753"),
             ("D-B  filtro de vivienda propia", "1101"),
             ("D-D  recorte dentro del bucle", "5671-5722"),
             ("D-E  piso numérico sobre gastos nulos", "1766 y análogas"),
             ("D-F  composición de la categoría Pan", "1979-1990"),
             ("D-H  controles de costo sin dimensión sectorial", "6182-6209")]:
    print(f"  {d:<48} l.{l}")

  ok  aradillas_core.py
  ok  datos_base.py
  ok  datos_2014.py
  ok  datos_2022.py
  ok  Replica_COFECE/CD/programa_ENIGH_2014.g

Referencias de línea en programa_ENIGH_2014.g (§2):
  D-A  estimación en una sola etapa OLS            l.2452-5753
  D-B  filtro de vivienda propia                   l.1101
  D-D  recorte dentro del bucle                    l.5671-5722
  D-E  piso numérico sobre gastos nulos            l.1766 y análogas
  D-F  composición de la categoría Pan             l.1979-1990
  D-H  controles de costo sin dimensión sectorial  l.6182-6209
